In [3]:
import pandas as pd

DATA_DIR = r'C:\Projects\ecommerce-bi-dashboard\data\raw'

orders = pd.read_csv(f'{DATA_DIR}\\olist_orders_dataset.csv')
items = pd.read_csv(f'{DATA_DIR}\\olist_order_items_dataset.csv')
customers = pd.read_csv(f'{DATA_DIR}\\olist_customers_dataset.csv')
products = pd.read_csv(f'{DATA_DIR}\\olist_products_dataset.csv')
sellers = pd.read_csv(f'{DATA_DIR}\\olist_sellers_dataset.csv')
payments = pd.read_csv(f'{DATA_DIR}\\olist_order_payments_dataset.csv')
reviews = pd.read_csv(f'{DATA_DIR}\\olist_order_reviews_dataset.csv')
category_translation = pd.read_csv(f'{DATA_DIR}\\product_category_name_translation.csv')

print("All files loaded successfully")
print(f"orders: {orders.shape}")
print(f"items: {items.shape}")
print(f"customers: {customers.shape}")
print(f"products: {products.shape}")
print(f"sellers: {sellers.shape}")
print(f"payments: {payments.shape}")
print(f"reviews: {reviews.shape}")
print(f"category_translation: {category_translation.shape}")

All files loaded successfully
orders: (99441, 8)
items: (112650, 7)
customers: (99441, 5)
products: (32951, 9)
sellers: (3095, 4)
payments: (103886, 5)
reviews: (99224, 7)
category_translation: (71, 2)


In [4]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders[date_cols].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [5]:
print(orders['order_status'].value_counts()) #how many orders didnt get delivered
print(f"\nOrders with no delivered_customer_date: {orders['order_delivered_customer_date'].isnull().sum()}")

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Orders with no delivered_customer_date: 2965


In [6]:
#product categorires fromportugese to english
products = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

products[['product_category_name', 'product_category_name_english']].head()
print(f"\nCategories with no English translation: {products['product_category_name_english'].isnull().sum()}")


Categories with no English translation: 623


In [7]:
delivered = orders[orders['order_status'] == 'delivered'].copy()

delivered['delivery_delay_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']
).dt.days

delivered['is_late'] = (delivered['delivery_delay_days'] > 0).astype(int)

print(f"Total delivered orders: {len(delivered)}")
print(f"Late deliveries: {delivered['is_late'].sum()} ({delivered['is_late'].mean()*100:.1f}%)")
print(f"\nDelay distribution:\n{delivered['delivery_delay_days'].describe()}")

Total delivered orders: 96478
Late deliveries: 6534 (6.8%)

Delay distribution:
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64


In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)

delivered.to_csv('../data/processed/orders_cleaned.csv', index=False)
products.to_csv('../data/processed/products_cleaned.csv', index=False)

print("Saved cleaned files to data/processed/")

Saved cleaned files to data/processed/
